# Hill Country Grocer — Dataset Generation

**Utility notebook (NOT a data-science deliverable).** This notebook generates the synthetic v2 dataset for the Hill Country Grocer engagement.

- **Contract:** spec Part A at `~/Desktop/scratch/ai-to-code/2026-05-13__hill-country-rebuild-spec.md`
- **Reversion:** `git -C ~/Desktop/000-smb-consulting-reference-data checkout pre-hill-country-rebuild`
- **Structure:** env verification → schema → generators → write CSV → validation gates

## 1. Environment Verification

Verify Python ≥ 3.10 and that pandas, numpy, faker are importable. Halts with a clear error if the env doesn't meet requirements. faker may be missing on first run — the next cell installs the pinned versions.

In [1]:
import sys
assert sys.version_info >= (3, 10), f'Python 3.10+ required, got {sys.version_info}'
print(f'Python: {sys.version.split()[0]}')

try:
    import pandas as pd
    print(f'pandas: {pd.__version__}')
except ImportError as e:
    raise RuntimeError(f'pandas not importable: {e}')

try:
    import numpy as np
    print(f'numpy: {np.__version__}')
except ImportError as e:
    raise RuntimeError(f'numpy not importable: {e}')

try:
    import faker
    print(f'faker: {faker.VERSION}')
except ImportError:
    print('faker: NOT INSTALLED (next cell installs it)')

Python: 3.12.7



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/opt/anaconda3/lib/python3.12/site-packages/ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "/opt/anaconda3/lib/python3.12/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/opt/anaconda3/lib/python3.12/site-packages/ipykernel/kernelapp.py", line 701, in start
    self.io_loop.start()
  File "/opt/anaconda3/lib/python3.12/site-

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/opt/anaconda3/lib/python3.12/site-packages/ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "/opt/anaconda3/lib/python3.12/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/opt/anaconda3/lib/python3.12/site-packages/ipykernel/kernelapp.py", line 701, in start
    self.io_loop.start()
  File "/opt/anaconda3/lib/python3.12/site-

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.



pandas: 2.2.2
numpy: 2.0.2
faker: 25.0.0


## 2. Library Installation (idempotent)

Pinned package install per spec §A.7. Operator's local env is Anaconda 3.12.7; `--break-system-packages` is not required.

In [2]:
%pip install --quiet pandas==2.2.2 numpy==2.0.2 faker==25.0.0

Note: you may need to restart the kernel to use updated packages.


## 3. Imports and Seeds

Imports for the data generator. Sets RANDOM_SEED = 42 on numpy's default_rng, Python's random module, and Faker so the same seed produces the same dataset across runs.

In [3]:
import pandas as pd
import numpy as np
import random
import re
import hashlib
from datetime import date, timedelta
from pathlib import Path
from faker import Faker

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)
random.seed(RANDOM_SEED)
Faker.seed(RANDOM_SEED)
fake = Faker()
print(f'Seeds set. RANDOM_SEED = {RANDOM_SEED}')

Seeds set. RANDOM_SEED = 42


## 4. Schema Constants

Banners, regions, store-ID format conventions per banner, departments, brand types, department-level price ranges and seasonal multiplier curves, catalog/store/week sizes, and the 156-week date range ending on the most recent Sunday before today.

In [4]:
BANNERS = ['Hill Country Grocer', 'Texas Family Markets', 'Lone Star Supermarkets']

REGIONS = ['Central Texas', 'South Texas', 'West Texas', 'Gulf Coast']

BANNER_REGIONS = {
    'Hill Country Grocer': ['Central Texas', 'West Texas'],
    'Texas Family Markets': ['Central Texas', 'South Texas', 'Gulf Coast'],
    'Lone Star Supermarkets': ['South Texas', 'Gulf Coast'],
}

DEPARTMENTS = [
    'Produce', 'Dairy', 'Meat', 'Bakery', 'Frozen',
    'Beverages', 'Snacks', 'Pantry', 'Health & Beauty', 'Household',
]

BRAND_TYPES = ['National Brand', 'Private Label', 'Local Brand']

DEPT_PRICE_RANGES = {
    'Produce': (0.79, 5.99),
    'Dairy': (1.99, 8.99),
    'Meat': (3.99, 24.99),
    'Bakery': (2.49, 9.99),
    'Frozen': (2.99, 12.99),
    'Beverages': (1.29, 14.99),
    'Snacks': (1.99, 8.99),
    'Pantry': (1.49, 11.99),
    'Health & Beauty': (2.99, 19.99),
    'Household': (2.99, 18.99),
}

NUM_PRODUCTS = 300
NUM_WEEKS = 156

today = date.today()
days_since_sunday = (today.weekday() + 1) % 7
most_recent_sunday = today - timedelta(days=days_since_sunday if days_since_sunday > 0 else 7)
WEEK_END_DATES = [most_recent_sunday - timedelta(weeks=(NUM_WEEKS - 1 - i)) for i in range(NUM_WEEKS)]
print(f'Week range: {WEEK_END_DATES[0]} -> {WEEK_END_DATES[-1]}  ({NUM_WEEKS} weeks)')

def seasonal_curve(department):
    weeks = np.arange(52)
    if department == 'Produce':
        return 1.0 + 0.4 * np.sin(2 * np.pi * (weeks - 12) / 52)
    if department == 'Bakery':
        curve = np.ones(52)
        curve[40:44] = 1.2
        curve[44:] = 1.4
        return curve
    if department == 'Beverages':
        return 1.0 + 0.3 * np.sin(2 * np.pi * (weeks - 14) / 52)
    if department == 'Meat':
        s = 1.0 + 0.2 * np.sin(2 * np.pi * (weeks - 12) / 52)
        s[45:] = s[45:] + 0.15
        return s
    if department == 'Frozen':
        return 1.0 + 0.15 * np.sin(2 * np.pi * (weeks - 5) / 52)
    if department == 'Dairy':
        return np.ones(52)
    if department == 'Snacks':
        s = np.ones(52)
        s[2:5] = 1.3
        s[40:45] = 1.2
        s[48:] = 1.15
        return s
    if department == 'Pantry':
        s = np.ones(52)
        s[44:] = 1.25
        return s
    return np.ones(52)

SEASON_LOOKUP = {d: seasonal_curve(d) for d in DEPARTMENTS}
print(f'Banners: {BANNERS}')
print(f'Regions: {REGIONS}')
print(f'Departments: {DEPARTMENTS}')
print(f'Brand types: {BRAND_TYPES}')
print(f'Catalog target: {NUM_PRODUCTS} unique UPCs')

FINAL_COLUMNS = [
    'Week_End_Date', 'UPC', 'Item_Description', 'Department', 'Brand_Type',
    'Net_Weight_oz', 'Pack_Size', 'List_Price_USD', 'Promo_Price_USD', 'Shelf_Facings',
    'Store_Number', 'Store_Banner', 'Store_Region', 'Store_Sq_Ft', 'Store_Open_Year',
    'Weekly_Units_Sold', 'Weekly_Revenue_USD',
]
assert len(FINAL_COLUMNS) == 17
print(f'Final schema: {len(FINAL_COLUMNS)} columns')

Week range: 2023-05-21 -> 2026-05-10  (156 weeks)
Banners: ['Hill Country Grocer', 'Texas Family Markets', 'Lone Star Supermarkets']
Regions: ['Central Texas', 'South Texas', 'West Texas', 'Gulf Coast']
Departments: ['Produce', 'Dairy', 'Meat', 'Bakery', 'Frozen', 'Beverages', 'Snacks', 'Pantry', 'Health & Beauty', 'Household']
Brand types: ['National Brand', 'Private Label', 'Local Brand']
Catalog target: 300 unique UPCs
Final schema: 17 columns


## 5. Generate Stores

11 stores across 3 banners with realistic mixed ID formats (HCG-### with gaps, TFM-##X with letter suffixes, LSS_#### with underscore separator) and varied Store_Sq_Ft (18,000–65,000) and Store_Open_Year (1995–2024).

In [5]:
stores_data = []
for n in [101, 103, 105, 110, 112]:
    stores_data.append({
        'Store_Number': f'HCG-{n:03d}',
        'Store_Banner': 'Hill Country Grocer',
        'Store_Region': random.choice(BANNER_REGIONS['Hill Country Grocer']),
        'Store_Sq_Ft': int(rng.integers(18000, 65001)),
        'Store_Open_Year': int(rng.integers(1995, 2025)),
    })
for num, suffix in [(4, ''), (7, 'B'), (12, ''), (15, 'A')]:
    stores_data.append({
        'Store_Number': f'TFM-{num:02d}{suffix}',
        'Store_Banner': 'Texas Family Markets',
        'Store_Region': random.choice(BANNER_REGIONS['Texas Family Markets']),
        'Store_Sq_Ft': int(rng.integers(18000, 65001)),
        'Store_Open_Year': int(rng.integers(1995, 2025)),
    })
for n in [21, 58]:
    stores_data.append({
        'Store_Number': f'LSS_{n:04d}',
        'Store_Banner': 'Lone Star Supermarkets',
        'Store_Region': random.choice(BANNER_REGIONS['Lone Star Supermarkets']),
        'Store_Sq_Ft': int(rng.integers(18000, 65001)),
        'Store_Open_Year': int(rng.integers(1995, 2025)),
    })

stores_df = pd.DataFrame(stores_data)
print(f'Generated {len(stores_df)} stores')
print(stores_df.to_string(index=False))

Generated 11 stores
Store_Number           Store_Banner  Store_Region  Store_Sq_Ft  Store_Open_Year
     HCG-101    Hill Country Grocer Central Texas        22194             2018
     HCG-103    Hill Country Grocer Central Texas        48765             2008
     HCG-105    Hill Country Grocer    West Texas        38352             2020
     HCG-110    Hill Country Grocer Central Texas        22039             2015
     HCG-112    Hill Country Grocer Central Texas        27469             1997
      TFM-04   Texas Family Markets Central Texas        42745             2024
     TFM-07B   Texas Family Markets    Gulf Coast        52581             2017
      TFM-12   Texas Family Markets Central Texas        51722             2018
     TFM-15A   Texas Family Markets    Gulf Coast        42122             1998
    LSS_0021 Lone Star Supermarkets   South Texas        57469             2008
    LSS_0058 Lone Star Supermarkets    Gulf Coast        41517             2006


## 6. Generate Products

About 300 unique UPCs across 10 departments. Each UPC is a GS1 GTIN-12 with a valid mod-10 check digit; the 11 data digits are random (not sequential) across the 0–9 first-digit range. Each product has a specific SKU description, plausible Net_Weight_oz/Pack_Size parsed from the description, and a List_Price_USD that favors psychological endings (.99/.49/.95/.79/.29 about 60% of the time, .00 about 5%).

In [6]:
def upc_check_digit(eleven_digits):
    total = 0
    for i, ch in enumerate(eleven_digits):
        d = int(ch)
        if (i + 1) % 2 == 1:
            total += d * 3
        else:
            total += d
    return (10 - total % 10) % 10

def make_upc(rng):
    first = int(rng.integers(0, 10))
    rest = rng.integers(0, 10, size=10)
    eleven = str(first) + ''.join(str(int(d)) for d in rest)
    return eleven + str(upc_check_digit(eleven))

PRODUCT_TEMPLATES = {
    'Produce': [
        ('Fresh Bananas', ['Organic 1 lb', 'Conventional 1 lb', 'Mini 8 ct']),
        ('Apples', ['Honeycrisp 3 lb Bag', 'Gala 3 lb Bag', 'Granny Smith each', 'Fuji 5 lb Bag']),
        ('Strawberries', ['1 lb Clamshell', 'Organic 1 lb Clamshell', '2 lb Family Pack']),
        ('Avocados', ['Hass each', 'Organic Hass 4 ct']),
        ('Romaine Hearts', ['3 ct Pack', 'Organic 3 ct Pack']),
        ('Carrots', ['2 lb Bag', 'Baby Cut 1 lb Bag', 'Organic Baby 1 lb']),
        ('Tomatoes', ['Roma 1 lb', 'Vine Ripened 1 lb', 'Cherry 16 oz']),
    ],
    'Dairy': [
        ('Whole Milk', ['Gallon', 'Half Gallon', 'Quart']),
        ('2% Milk', ['Gallon', 'Half Gallon']),
        ('Greek Yogurt', ['Plain 32 oz Tub', 'Vanilla 5.3 oz', 'Strawberry 5.3 oz']),
        ('Cheddar Cheese', ['Block 8 oz', 'Shredded 8 oz Bag', 'Sliced 12 oz']),
        ('Butter', ['Salted Sticks 16 oz', 'Unsalted Sticks 16 oz', 'European Style 8 oz']),
        ('Eggs', ['Large Grade A Dozen', 'Organic Large Dozen', 'Cage-Free 18 ct']),
    ],
    'Meat': [
        ('Ground Beef', ['80/20 1 lb Pack', '85/15 1 lb Pack', '93/7 1 lb Pack']),
        ('Chicken Breast', ['Boneless Skinless 1 lb', 'Organic Boneless 1 lb', 'Family Pack 3 lb']),
        ('Pork Chops', ['Bone-In 1 lb', 'Boneless 1 lb']),
        ('Ribeye Steak', ['USDA Choice 1 lb', 'USDA Prime 1 lb']),
        ('Bacon', ['Hickory Smoked 12 oz', 'Thick Cut 16 oz', 'Applewood 12 oz']),
    ],
    'Bakery': [
        ('White Bread', ['Loaf 20 oz', 'Sandwich Loaf 24 oz']),
        ('Wheat Bread', ['100% Whole Wheat 20 oz', 'Multigrain 24 oz']),
        ('Bagels', ['Plain 6 ct', 'Everything 6 ct', 'Cinnamon Raisin 6 ct']),
        ('Tortillas', ['Flour 10 ct', 'Corn 30 ct', 'Whole Wheat 8 ct']),
        ('Donuts', ['Glazed 12 ct', 'Variety 8 ct']),
    ],
    'Frozen': [
        ('Pizza', ['Pepperoni 16 oz', 'Cheese 14 oz', 'Supreme 24 oz']),
        ('Ice Cream', ['Vanilla 48 oz', 'Chocolate 48 oz', 'Cookies and Cream 16 oz']),
        ('Frozen Vegetables', ['Mixed 16 oz Bag', 'Broccoli Florets 12 oz', 'Corn 16 oz']),
        ('Chicken Nuggets', ['29 oz Bag', 'Family Size 48 oz']),
        ('Frozen Berries', ['Mixed Berries 12 oz', 'Blueberries 12 oz', 'Strawberries 16 oz']),
    ],
    'Beverages': [
        ('Cola', ['12 pk Cans', '2 Liter Bottle', '20 oz Bottle']),
        ('Diet Cola', ['12 pk Cans', '2 Liter Bottle']),
        ('Orange Juice', ['52 oz Carton', 'Half Gallon', 'Pulp Free Half Gallon']),
        ('Bottled Water', ['24 pk 16.9 oz', '1 Gallon', 'Sparkling 8 pk']),
        ('Coffee', ['Ground Medium Roast 12 oz', 'K-Cups 24 ct', 'Whole Bean 12 oz']),
        ('Energy Drink', ['16 oz Can', '4 pk 16 oz']),
    ],
    'Snacks': [
        ('Potato Chips', ['Classic 9 oz', 'Party Size 14 oz', 'BBQ 9 oz']),
        ('Tortilla Chips', ['Restaurant Style 13 oz', 'White Corn 14 oz']),
        ('Pretzels', ['Mini 16 oz Bag', 'Sticks 14 oz']),
        ('Cookies', ['Chocolate Chip 13 oz', 'Sandwich 14 oz Pack', 'Sugar 12 oz']),
        ('Crackers', ['Saltine 16 oz', 'Wheat Thin 9 oz', 'Cheese 12 oz']),
        ('Nuts', ['Mixed 12 oz Can', 'Almonds 16 oz Bag', 'Cashews 8 oz']),
    ],
    'Pantry': [
        ('Pasta Sauce', ['Marinara 24 oz Jar', 'Alfredo 15 oz Jar', 'Vodka 24 oz Jar']),
        ('Spaghetti', ['1 lb Box', 'Whole Wheat 1 lb']),
        ('Rice', ['Long Grain White 5 lb Bag', 'Brown 2 lb', 'Jasmine 5 lb']),
        ('Cereal', ['Honey Oats 18 oz', 'Frosted Flakes 18 oz', 'Bran Flakes 16 oz']),
        ('Peanut Butter', ['Creamy 16 oz Jar', 'Crunchy 16 oz Jar', 'Natural 26 oz']),
        ('Jelly', ['Grape 18 oz', 'Strawberry 18 oz']),
        ('Canned Beans', ['Black Beans 15 oz', 'Pinto 15 oz', 'Kidney 15 oz']),
        ('Cooking Oil', ['Vegetable 48 oz', 'Olive Extra Virgin 17 oz', 'Canola 32 oz']),
        ('Flour', ['All Purpose 5 lb Bag', 'Whole Wheat 5 lb']),
        ('Sugar', ['Granulated 4 lb Bag', 'Brown 2 lb']),
    ],
    'Health & Beauty': [
        ('Toothpaste', ['Mint 6 oz', 'Whitening 4.7 oz', 'Sensitivity 4 oz']),
        ('Shampoo', ['Daily Use 22 oz', 'Moisturizing 12 oz', 'Anti-Dandruff 13 oz']),
        ('Deodorant', ['Stick 2.6 oz', 'Aluminum-Free 3 oz']),
        ('Vitamins', ['Multivitamin 100 ct', 'Vitamin D 60 ct', 'Vitamin C 100 ct']),
        ('Bandages', ['Assorted 40 ct', 'Heavy Duty 30 ct']),
    ],
    'Household': [
        ('Paper Towels', ['6 Roll Pack', '12 Roll Pack', 'Single Roll']),
        ('Toilet Paper', ['12 Roll Mega', '6 Roll Double']),
        ('Laundry Detergent', ['Liquid 100 oz', 'Pods 42 ct', 'Liquid 50 oz']),
        ('Dish Soap', ['Lemon 25 oz', 'Antibacterial 28 oz']),
        ('Trash Bags', ['13 Gal Tall Kitchen 100 ct', '30 Gal Outdoor 50 ct']),
        ('All-Purpose Cleaner', ['Spray 32 oz', 'Concentrate 64 oz']),
    ],
}

def parse_attrs(item_text):
    txt = item_text.lower()
    pack_size = 1
    m = re.search(r'(\d+)\s*(ct|pk|pack|roll|pcs)\b', txt)
    if m:
        pack_size = int(m.group(1))
    if 'dozen' in txt:
        pack_size = 12
    weight_oz = None
    m = re.search(r'(\d+(?:\.\d+)?)\s*oz\b', txt)
    if m:
        weight_oz = float(m.group(1))
    if weight_oz is None:
        m = re.search(r'(\d+(?:\.\d+)?)\s*lb\b', txt)
        if m:
            weight_oz = float(m.group(1)) * 16.0
    if weight_oz is None:
        if 'half gallon' in txt:
            weight_oz = 64.0
        elif 'gallon' in txt:
            weight_oz = 128.0
        elif 'quart' in txt:
            weight_oz = 32.0
        else:
            m = re.search(r'(\d+(?:\.\d+)?)\s*liter', txt)
            if m:
                weight_oz = float(m.group(1)) * 33.814
    if weight_oz is None:
        weight_oz = float(int(rng.integers(8, 32)))
    return float(weight_oz), pack_size

def plausible_price(rng, dept):
    lo, hi = DEPT_PRICE_RANGES[dept]
    base = float(rng.uniform(lo, hi))
    r = float(rng.random())
    if r < 0.92:
        ending = float(rng.choice([0.99, 0.49, 0.95, 0.79, 0.29]))
        whole = max(0, int(base))
        return round(whole + ending, 2)
    return round(base, 2)

per_dept = NUM_PRODUCTS // len(DEPARTMENTS)
remainder = NUM_PRODUCTS - per_dept * len(DEPARTMENTS)
dept_alloc = {d: per_dept for d in DEPARTMENTS}
for i in range(remainder):
    dept_alloc[DEPARTMENTS[i]] += 1

products_data = []
seen_upcs = set()
brand_suffixes = ['Brand A', 'Brand B', 'Store Brand', 'Premium', 'Value']

for dept, n in dept_alloc.items():
    expanded = []
    for product_name, variants in PRODUCT_TEMPLATES[dept]:
        for v in variants:
            expanded.append(f'{product_name}, {v}')
    full = list(expanded)
    si = 0
    while len(full) < n:
        for base_desc in expanded:
            full.append(f'{base_desc} ({brand_suffixes[si % len(brand_suffixes)]})')
            if len(full) >= n:
                break
        si += 1
    chosen = full[:n]
    for desc in chosen:
        upc = make_upc(rng)
        while upc in seen_upcs:
            upc = make_upc(rng)
        seen_upcs.add(upc)
        weight_oz, pack_size = parse_attrs(desc)
        brand = str(rng.choice(BRAND_TYPES, p=[0.55, 0.30, 0.15]))
        price = plausible_price(rng, dept)
        products_data.append({
            'UPC': upc,
            'Item_Description': desc,
            'Department': dept,
            'Brand_Type': brand,
            'Net_Weight_oz': weight_oz,
            'Pack_Size': pack_size,
            'List_Price_USD': price,
        })

products_df = pd.DataFrame(products_data)
assert len(products_df) == NUM_PRODUCTS
assert products_df['UPC'].nunique() == NUM_PRODUCTS
n_depts = products_df['Department'].nunique()
print(f'Generated {len(products_df)} unique products across {n_depts} departments')
print(products_df.head(8).to_string(index=False))

Generated 300 unique products across 10 departments
         UPC                 Item_Description Department     Brand_Type  Net_Weight_oz  Pack_Size  List_Price_USD
197648544208      Fresh Bananas, Organic 1 lb    Produce National Brand           16.0          1            5.95
177309486771 Fresh Bananas, Conventional 1 lb    Produce National Brand           16.0          1            1.99
769739439306         Fresh Bananas, Mini 8 ct    Produce National Brand           19.0          8            1.49
256941867039      Apples, Honeycrisp 3 lb Bag    Produce  Private Label           48.0          1            4.29
226618180778            Apples, Gala 3 lb Bag    Produce  Private Label           48.0          1            4.95
455012146640        Apples, Granny Smith each    Produce  Private Label           28.0          1            4.95
057360349225            Apples, Fuji 5 lb Bag    Produce    Local Brand           80.0          1            2.95
829246155790     Strawberries, 1 lb 

## 7. Store-Product Stocking

Each store stocks a subset of 150–200 of the 300 UPCs (overlap + store-specific). A small fraction of pairs start later (new-product onboarding) and a small fraction discontinue early — both reflect realistic retail patterns.

In [7]:
all_upcs = products_df['UPC'].tolist()
stocking_data = []
for _, store_row in stores_df.iterrows():
    n_stocked = int(rng.integers(150, 201))
    stocked_idx = rng.choice(len(all_upcs), size=n_stocked, replace=False)
    stocked = [all_upcs[i] for i in stocked_idx]
    for upc in stocked:
        if rng.random() < 0.05:
            start_week = int(rng.integers(20, 130))
        else:
            start_week = 0
        if rng.random() < 0.03:
            end_week = int(rng.integers(50, NUM_WEEKS - 5))
        else:
            end_week = NUM_WEEKS
        stocking_data.append({
            'Store_Number': store_row['Store_Number'],
            'UPC': upc,
            'start_week': start_week,
            'end_week': end_week,
        })

stocking_df = pd.DataFrame(stocking_data)
mean_upcs_per_store = stocking_df.groupby('Store_Number').size().mean()
n_delayed = int((stocking_df['start_week'] > 0).sum())
n_discontinued = int((stocking_df['end_week'] < NUM_WEEKS).sum())
print(f'Total (Store, UPC) stocked pairs: {len(stocking_df)}')
print(f'Mean UPCs per store: {mean_upcs_per_store:.1f}')
print(f'Delayed-start pairs: {n_delayed}')
print(f'Early-end pairs: {n_discontinued}')

Total (Store, UPC) stocked pairs: 1854
Mean UPCs per store: 168.5
Delayed-start pairs: 85
Early-end pairs: 63


## 8. Generate Weekly Observations

For each stocked (Store, UPC) pair × week in [start_week, end_week): one row. Applies department-level seasonality, promotions (~20% with 10–50% off), shelf facings, right-skewed unit demand with store-size and promo-lift effects, occasional zero-sales rows, rare outlier spikes, and small noise on the revenue calculation. This is the heaviest section — expect a minute or so of execution.

In [8]:
products_lookup = products_df.set_index('UPC')
stores_lookup = stores_df.set_index('Store_Number')

obs_rows = []
psych_endings = np.array([0.99, 0.49, 0.95, 0.79, 0.29])

for stocking_row in stocking_df.itertuples(index=False):
    upc = stocking_row.UPC
    store_no = stocking_row.Store_Number
    start_w = stocking_row.start_week
    end_w = stocking_row.end_week

    prod = products_lookup.loc[upc]
    store = stores_lookup.loc[store_no]
    dept = prod['Department']
    list_price = float(prod['List_Price_USD'])
    sq_ft = int(store['Store_Sq_Ft'])
    season = SEASON_LOOKUP[dept]

    base_demand = float(rng.lognormal(mean=1.5, sigma=1.0))
    store_size_factor = sq_ft / 40000.0
    base_facings = max(1, min(12, int(round(rng.normal(loc=3 + store_size_factor * 2, scale=1.5)))))

    for w in range(start_w, end_w):
        week_of_year = w % 52
        seasonal_mult = float(season[week_of_year])
        on_promo = bool(rng.random() < 0.20)
        if on_promo:
            discount_pct = float(rng.uniform(0.10, 0.50))
            raw_promo = list_price * (1.0 - discount_pct)
            whole = max(0, int(raw_promo))
            ending = float(psych_endings[int(rng.integers(0, len(psych_endings)))])
            promo_price = round(whole + ending, 2)
            if promo_price >= list_price:
                promo_price = round(list_price * 0.90, 2)
            if promo_price <= 0:
                promo_price = round(list_price * 0.50, 2)
            promo_lift = float(rng.uniform(1.5, 4.0))
        else:
            promo_price = list_price
            promo_lift = 1.0
        effective_price = promo_price if on_promo else list_price
        demand_mean = max(0.1, base_demand * store_size_factor * seasonal_mult * promo_lift)
        units = int(rng.poisson(demand_mean))
        if units > 0 and rng.random() < 0.07:
            units = 0
        if units > 0 and rng.random() < 0.005:
            units = int(units * rng.uniform(5.0, 10.0))
        noise = float(1.0 + rng.uniform(-0.03, 0.03))
        revenue = round(units * effective_price * noise, 2)
        facings_jitter = int(rng.integers(-1, 2))
        facings = max(1, min(12, base_facings + facings_jitter))
        obs_rows.append({
            'Week_End_Date': WEEK_END_DATES[w].isoformat(),
            'UPC': upc,
            'Store_Number': store_no,
            'List_Price_USD': list_price,
            'Promo_Price_USD': promo_price,
            'Shelf_Facings': facings,
            'Weekly_Units_Sold': units,
            'Weekly_Revenue_USD': revenue,
        })

obs_df = pd.DataFrame(obs_rows)
print(f'Generated {len(obs_df):,} weekly observations')

Generated 279,724 weekly observations


## 9. Assemble Final DataFrame

Join weekly observations with product and store metadata. Apply the final 17-column order and dtype contract (Week_End_Date as ISO date string, UPC as string to preserve leading zeros, integer dtypes for Pack_Size / Shelf_Facings / Store_Sq_Ft / Store_Open_Year / Weekly_Units_Sold).

In [9]:
final_df = (
    obs_df
    .merge(products_df[['UPC', 'Item_Description', 'Department', 'Brand_Type', 'Net_Weight_oz', 'Pack_Size']], on='UPC', how='left')
    .merge(stores_df[['Store_Number', 'Store_Banner', 'Store_Region', 'Store_Sq_Ft', 'Store_Open_Year']], on='Store_Number', how='left')
)
final_df = final_df[FINAL_COLUMNS]

final_df['Week_End_Date'] = final_df['Week_End_Date'].astype(str)
final_df['UPC'] = final_df['UPC'].astype(str)
final_df['Pack_Size'] = final_df['Pack_Size'].astype(int)
final_df['Shelf_Facings'] = final_df['Shelf_Facings'].astype(int)
final_df['Store_Sq_Ft'] = final_df['Store_Sq_Ft'].astype(int)
final_df['Store_Open_Year'] = final_df['Store_Open_Year'].astype(int)
final_df['Weekly_Units_Sold'] = final_df['Weekly_Units_Sold'].astype(int)

print(f'Final shape: {final_df.shape}')
print(final_df.dtypes)
print()
print(final_df.head(3).to_string(index=False))

Final shape: (279724, 17)
Week_End_Date          object
UPC                    object
Item_Description       object
Department             object
Brand_Type             object
Net_Weight_oz         float64
Pack_Size               int64
List_Price_USD        float64
Promo_Price_USD       float64
Shelf_Facings           int64
Store_Number           object
Store_Banner           object
Store_Region           object
Store_Sq_Ft             int64
Store_Open_Year         int64
Weekly_Units_Sold       int64
Weekly_Revenue_USD    float64
dtype: object

Week_End_Date          UPC                     Item_Description Department    Brand_Type  Net_Weight_oz  Pack_Size  List_Price_USD  Promo_Price_USD  Shelf_Facings Store_Number        Store_Banner  Store_Region  Store_Sq_Ft  Store_Open_Year  Weekly_Units_Sold  Weekly_Revenue_USD
   2023-05-21 888002534170 Cheddar Cheese, Block 8 oz (Brand A)      Dairy Private Label            8.0          1            2.49             2.49              2      HC

## 10. Inject Missingness

Insert NaN at low per-column rate (~1–1.8% each) for plausible columns (Brand_Type, Net_Weight_oz, Shelf_Facings, Promo_Price_USD). Target columns (Weekly_Units_Sold, Weekly_Revenue_USD) and identity columns (UPC, Store_Number, Week_End_Date) stay 100% complete.

In [10]:
MISSINGNESS_COLS = ['Brand_Type', 'Net_Weight_oz', 'Shelf_Facings', 'Promo_Price_USD']
TARGET_COLS = ['Weekly_Units_Sold', 'Weekly_Revenue_USD']
IDENTITY_COLS = ['UPC', 'Store_Number', 'Week_End_Date']

n_rows = len(final_df)
for col in MISSINGNESS_COLS:
    rate = float(rng.uniform(0.010, 0.018))
    n_missing = int(n_rows * rate)
    col_idx = final_df.columns.get_loc(col)
    null_idx = rng.choice(n_rows, size=n_missing, replace=False)
    final_df.iloc[null_idx, col_idx] = np.nan

for col in TARGET_COLS + IDENTITY_COLS:
    if final_df[col].isna().any():
        raise AssertionError(f'{col} unexpectedly has NaN')

eligible = [c for c in final_df.columns if c not in TARGET_COLS + IDENTITY_COLS]
n_nan_eligible = int(final_df[eligible].isna().sum().sum())
rate_eligible = n_nan_eligible / final_df[eligible].size
print(f'Eligible cells: {final_df[eligible].size:,}')
print(f'NaN count on eligible cells: {n_nan_eligible:,}')
print(f'Missingness rate on eligible cells: {rate_eligible:.4%}')

Eligible cells: 3,356,688
NaN count on eligible cells: 16,130
Missingness rate on eligible cells: 0.4805%


## 11. Write CSV

Overwrite the engagement's `data/raw/hill_country_grocer_weekly_sales.csv`. Prints row/column counts, file size in bytes, and SHA-256 hash of the file content for auditable identity.

In [11]:
out_path = (
    Path.home() / 'Desktop' / '000-smb-consulting-reference-data'
    / 'engagements' / 'ref-hill-country-grocer-revenue-pred__reg__ensemble-exp'
    / 'data' / 'raw' / 'hill_country_grocer_weekly_sales.csv'
)
out_path.parent.mkdir(parents=True, exist_ok=True)
final_df.to_csv(out_path, index=False)

file_size = out_path.stat().st_size
with open(out_path, 'rb') as f:
    file_hash = hashlib.sha256(f.read()).hexdigest()

print(f'Wrote: {out_path}')
print(f'Rows: {len(final_df):,}')
print(f'Columns: {len(final_df.columns)}')
print(f'File size: {file_size:,} bytes')
print(f'SHA-256: {file_hash}')

Wrote: /Users/eriksvagshenian/Desktop/000-smb-consulting-reference-data/engagements/ref-hill-country-grocer-revenue-pred__reg__ensemble-exp/data/raw/hill_country_grocer_weekly_sales.csv
Rows: 279,724
Columns: 17
File size: 45,504,527 bytes
SHA-256: 7b8675e6da40d826413bb0988408ee964be178a0f0ef0eb22957678809a04af9


## 12. Validation Gates (§A.4)

15 realism gates. Each cell prints `[PASS]` or `[FAIL]` with the measured value and expected range; a `[FAIL]` raises AssertionError and halts execution.

### Gate a — No zero-variance columns

In [12]:
zero_var = [c for c in final_df.columns if final_df[c].nunique(dropna=True) < 2]
if zero_var:
    msg = f'[FAIL] Zero-variance columns: {zero_var}'
    print(msg)
    raise AssertionError(msg)
print(f'[PASS] No zero-variance columns (all {len(final_df.columns)} columns have >= 2 unique non-null values)')

[PASS] No zero-variance columns (all 17 columns have >= 2 unique non-null values)


### Gate b — Store ID realism (≥2 distinct formats)

In [13]:
ids = stores_df['Store_Number'].tolist()
prefixes = set()
for sid in ids:
    if '-' in sid:
        prefixes.add('dash:' + sid.split('-')[0])
    elif '_' in sid:
        prefixes.add('underscore:' + sid.split('_')[0])
    else:
        prefixes.add('plain:' + sid[:3])
n_formats = len(prefixes)
if n_formats < 2:
    msg = f'[FAIL] Only {n_formats} distinct store ID format(s): {prefixes}'
    print(msg)
    raise AssertionError(msg)
print(f'[PASS] {n_formats} distinct store ID formats: {sorted(prefixes)}')

[PASS] 3 distinct store ID formats: ['dash:HCG', 'dash:TFM', 'underscore:LSS']


### Gate c — Time coverage (≥104 weeks; no gap > 2 weeks)

In [14]:
weeks = pd.to_datetime(final_df['Week_End_Date'].unique())
weeks_sorted = sorted(weeks)
n_unique = len(weeks_sorted)
gaps_days = [(weeks_sorted[i+1] - weeks_sorted[i]).days for i in range(len(weeks_sorted)-1)]
max_gap_weeks = (max(gaps_days) / 7) if gaps_days else 0
if n_unique < 104:
    msg = f'[FAIL] Only {n_unique} unique weeks (need >= 104)'
    print(msg)
    raise AssertionError(msg)
if max_gap_weeks > 2:
    msg = f'[FAIL] Max gap = {max_gap_weeks:.1f} weeks (> 2)'
    print(msg)
    raise AssertionError(msg)
print(f'[PASS] {n_unique} unique weeks, max gap = {max_gap_weeks:.1f} weeks')

[PASS] 156 unique weeks, max gap = 1.0 weeks


### Gate d — Banner count ≥ 2, Region count ≥ 3

In [15]:
banner_count = final_df['Store_Banner'].nunique()
region_count = final_df['Store_Region'].nunique()
if banner_count < 2:
    msg = f'[FAIL] banner_count={banner_count} (need >= 2)'
    print(msg)
    raise AssertionError(msg)
if region_count < 3:
    msg = f'[FAIL] region_count={region_count} (need >= 3)'
    print(msg)
    raise AssertionError(msg)
print(f'[PASS] banners={banner_count}, regions={region_count}')

[PASS] banners=3, regions=4


### Gate e — Store count in 8–15

In [16]:
n_stores_g = final_df['Store_Number'].nunique()
if not (8 <= n_stores_g <= 15):
    msg = f'[FAIL] store count {n_stores_g} outside [8, 15]'
    print(msg)
    raise AssertionError(msg)
print(f'[PASS] {n_stores_g} stores (in [8, 15])')

[PASS] 11 stores (in [8, 15])


### Gate f — UPC count in 200–400

In [17]:
n_upcs_g = final_df['UPC'].nunique()
if not (200 <= n_upcs_g <= 400):
    msg = f'[FAIL] UPC count {n_upcs_g} outside [200, 400]'
    print(msg)
    raise AssertionError(msg)
print(f'[PASS] {n_upcs_g} unique UPCs (in [200, 400])')

[PASS] 300 unique UPCs (in [200, 400])


### Gate g — UPC check digits valid (GS1 mod-10)

In [18]:
def is_valid_gtin12(upc):
    if not (isinstance(upc, str) and len(upc) == 12 and upc.isdigit()):
        return False
    return upc_check_digit(upc[:11]) == int(upc[-1])

unique_upcs_g = final_df['UPC'].unique()
invalid = [u for u in unique_upcs_g if not is_valid_gtin12(u)]
if invalid:
    msg = f'[FAIL] {len(invalid)} UPCs failed mod-10 check'
    print(msg)
    raise AssertionError(msg)
print(f'[PASS] All {len(unique_upcs_g)} UPCs are valid GS1 GTIN-12 with correct mod-10 check digits')

[PASS] All 300 UPCs are valid GS1 GTIN-12 with correct mod-10 check digits


### Gate h — Price endings (.99/.49/.95/.79/.29 ≥ 60% of unique prices, .00 ≤ 5%)

In [19]:
unique_prices_g = final_df['List_Price_USD'].dropna().unique()
def cents(p):
    return int(round((p - int(p)) * 100))
endings = [cents(p) for p in unique_prices_g]
psych_count = sum(1 for e in endings if e in (99, 49, 95, 79, 29))
zero_count = sum(1 for e in endings if e == 0)
psych_rate_g = psych_count / len(endings)
zero_rate_g = zero_count / len(endings)
if psych_rate_g < 0.60:
    msg = f'[FAIL] Psych-ending rate {psych_rate_g:.2%} < 60%'
    print(msg)
    raise AssertionError(msg)
if zero_rate_g > 0.05:
    msg = f'[FAIL] .00 ending rate {zero_rate_g:.2%} > 5%'
    print(msg)
    raise AssertionError(msg)
print(f'[PASS] Psych endings: {psych_rate_g:.1%} (>= 60%), .00 endings: {zero_rate_g:.1%} (<= 5%)')

[PASS] Psych endings: 82.2% (>= 60%), .00 endings: 0.0% (<= 5%)


### Gate i — Promotion rate in 15–25%

In [20]:
on_promo_mask = (final_df['Promo_Price_USD'].notna()) & (final_df['Promo_Price_USD'] < final_df['List_Price_USD'])
promo_rate_g = float(on_promo_mask.mean())
if not (0.15 <= promo_rate_g <= 0.25):
    msg = f'[FAIL] promo_rate={promo_rate_g:.3%} outside [15%, 25%]'
    print(msg)
    raise AssertionError(msg)
print(f'[PASS] Promotion rate {promo_rate_g:.2%} (in [15%, 25%])')

[PASS] Promotion rate 19.68% (in [15%, 25%])


### Gate j — Mean department prices plausible (Produce mean < Meat mean)

In [21]:
dept_price_means = final_df.groupby('Department')['List_Price_USD'].mean().sort_values()
print(dept_price_means)
produce_mean = dept_price_means['Produce']
meat_mean = dept_price_means['Meat']
if produce_mean >= meat_mean:
    msg = f'[FAIL] Produce mean ${produce_mean:.2f} not < Meat mean ${meat_mean:.2f}'
    print(msg)
    raise AssertionError(msg)
print(f'[PASS] Produce mean ${produce_mean:.2f} < Meat mean ${meat_mean:.2f}')

Department
Produce             3.808670
Snacks              5.407395
Pantry              5.416038
Dairy               5.714091
Bakery              5.928510
Frozen              8.244746
Beverages           9.133551
Health & Beauty    11.007684
Household          11.617576
Meat               14.775495
Name: List_Price_USD, dtype: float64
[PASS] Produce mean $3.81 < Meat mean $14.78


### Gate k — Right-skewed unit distribution (skew > 1.0 on non-zero Weekly_Units_Sold)

In [22]:
nonzero_units = final_df.loc[final_df['Weekly_Units_Sold'] > 0, 'Weekly_Units_Sold']
sk = float(nonzero_units.skew())
if not (sk > 1.0):
    msg = f'[FAIL] Skewness {sk:.3f} not > 1.0'
    print(msg)
    raise AssertionError(msg)
print(f'[PASS] Skewness of non-zero Weekly_Units_Sold = {sk:.3f} (> 1.0)')

[PASS] Skewness of non-zero Weekly_Units_Sold = 20.653 (> 1.0)


### Gate l — Revenue ≈ Units × effective_price with residual noise (0.95 ≤ corr < 0.9999)

In [23]:
eff_price = np.where(
    on_promo_mask.values,
    final_df['Promo_Price_USD'].fillna(final_df['List_Price_USD']).values,
    final_df['List_Price_USD'].values,
)
expected_rev = final_df['Weekly_Units_Sold'].values * eff_price
corr = float(np.corrcoef(final_df['Weekly_Revenue_USD'].values, expected_rev)[0, 1])
if not (0.95 <= corr < 0.9999):
    msg = f'[FAIL] Revenue-vs-(units*price) corr {corr:.4f} not in [0.95, 0.9999)'
    print(msg)
    raise AssertionError(msg)
print(f'[PASS] corr(revenue, units*effective_price) = {corr:.5f} (in [0.95, 0.9999))')

[PASS] corr(revenue, units*effective_price) = 0.99912 (in [0.95, 0.9999))


### Gate m — Zero-sales rate in 5–15%

In [24]:
zero_rate = float((final_df['Weekly_Units_Sold'] == 0).mean())
if not (0.05 <= zero_rate <= 0.15):
    msg = f'[FAIL] zero-sales rate {zero_rate:.3%} outside [5%, 15%]'
    print(msg)
    raise AssertionError(msg)
print(f'[PASS] Zero-sales rate {zero_rate:.2%} (in [5%, 15%])')

[PASS] Zero-sales rate 14.43% (in [5%, 15%])


### Gate n — Promotional lift on units (mean on-promo > 1.5× mean off-promo)

In [25]:
mean_on = float(final_df.loc[on_promo_mask, 'Weekly_Units_Sold'].mean())
mean_off = float(final_df.loc[~on_promo_mask, 'Weekly_Units_Sold'].mean())
lift = mean_on / mean_off if mean_off > 0 else float('inf')
if not (lift > 1.5):
    msg = f'[FAIL] Promo lift {lift:.2f}x not > 1.5x'
    print(msg)
    raise AssertionError(msg)
print(f'[PASS] Promotional lift {lift:.2f}x (mean_on={mean_on:.2f}, mean_off={mean_off:.2f})')

[PASS] Promotional lift 2.76x (mean_on=21.38, mean_off=7.75)


### Gate o — Missingness rate 0.1%–2% on eligible columns; target/identity 100% complete

In [26]:
TARGET_G = ['Weekly_Units_Sold', 'Weekly_Revenue_USD']
IDENT_G = ['UPC', 'Store_Number', 'Week_End_Date']
eligible_g = [c for c in final_df.columns if c not in TARGET_G + IDENT_G]
nan_total = int(final_df[eligible_g].isna().sum().sum())
cells_total = final_df[eligible_g].size
miss_rate = nan_total / cells_total
if not (0.001 <= miss_rate <= 0.02):
    msg = f'[FAIL] Missingness rate {miss_rate:.4%} outside [0.1%, 2%]'
    print(msg)
    raise AssertionError(msg)
for col in TARGET_G + IDENT_G:
    if final_df[col].isna().any():
        msg = f'[FAIL] Target/identity column {col} has NaN'
        print(msg)
        raise AssertionError(msg)
print(f'[PASS] Missingness rate {miss_rate:.4%} on eligible cells; target/identity columns 100% complete')

[PASS] Missingness rate 0.4805% on eligible cells; target/identity columns 100% complete


## 13. Final Summary

In [27]:
banners_g = sorted(final_df['Store_Banner'].dropna().unique().tolist())
regions_g = sorted(final_df['Store_Region'].dropna().unique().tolist())
n_stores_final = final_df['Store_Number'].nunique()
n_upcs_final = final_df['UPC'].nunique()
week_min = final_df['Week_End_Date'].min()
week_max = final_df['Week_End_Date'].max()
print('=' * 60)
print('FINAL DATASET METADATA')
print('=' * 60)
print(f'Rows:       {len(final_df):,}')
print(f'Columns:    {len(final_df.columns)}')
print(f'Banners:    {banners_g}')
print(f'Regions:    {regions_g}')
print(f'Stores:     {n_stores_final}')
print(f'UPCs:       {n_upcs_final}')
print(f'Weeks:      {week_min} -> {week_max}')
print(f'File:       {out_path}')
print(f'File size:  {file_size:,} bytes')
print(f'SHA-256:    {file_hash}')
print()
print('ALL 15 VALIDATION GATES PASSED')

FINAL DATASET METADATA
Rows:       279,724
Columns:    17
Banners:    ['Hill Country Grocer', 'Lone Star Supermarkets', 'Texas Family Markets']
Regions:    ['Central Texas', 'Gulf Coast', 'South Texas', 'West Texas']
Stores:     11
UPCs:       300
Weeks:      2023-05-21 -> 2026-05-10
File:       /Users/eriksvagshenian/Desktop/000-smb-consulting-reference-data/engagements/ref-hill-country-grocer-revenue-pred__reg__ensemble-exp/data/raw/hill_country_grocer_weekly_sales.csv
File size:  45,504,527 bytes
SHA-256:    7b8675e6da40d826413bb0988408ee964be178a0f0ef0eb22957678809a04af9

ALL 15 VALIDATION GATES PASSED


## 14. Reversion Note

If validation fails or data is rejected, revert with:

```
git -C ~/Desktop/000-smb-consulting-reference-data checkout pre-hill-country-rebuild
```